# Classification — Predicting Disease Stage

Same 418-patient PBC dataset as Phase 1, different question: can we predict a
patient's histologic `Stage` (1-4) from their biomarkers, without a liver biopsy?

Step 1: load the data and look at the target — the `Stage` column — and at
how much data is actually missing across the other columns, since that will
shape every decision from here on.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/cirrhosis.csv")
df["Stage"].value_counts().sort_index()

## Missing values

In Phase 1 we already saw that 106/418 patients weren't randomized into the
trial — they were only observed. Let's see exactly which columns that affects,
across the whole dataset.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
(missing / len(df) * 100).round(1)

## Choosing features and handling missing data

We'll use actual biomarkers and clinical findings — not `Drug` (a treatment
assignment, not a biological measurement — and `Stage` is diagnosed by
biopsy *before* treatment starts, so `Drug` shouldn't predict it) and not
`N_Days`/`Status` (that's the survival outcome from Phase 1 — using it here
would leak future information into a prediction we should be able to make at
diagnosis time).

After selecting features, we drop any row missing `Stage` (the target) or
any of the chosen feature columns.

In [ ]:
feature_cols = [
    "Age", "Sex", "Ascites", "Hepatomegaly", "Spiders", "Edema",
    "Bilirubin", "Cholesterol", "Albumin", "Copper", "Alk_Phos",
    "SGOT", "Tryglicerides", "Platelets", "Prothrombin",
]

model_df = df[feature_cols + ["Stage"]].dropna()
print(model_df.shape)
model_df["Stage"].value_counts().sort_index()

**Result:** 276 patients remain. Note the class imbalance — `Stage 1` has only
12 patients vs. 94-111 for the others. We'll need to keep this in mind for the
train/test split (stratify) and when judging the model later (accuracy alone
would be misleading here).

## Encoding categorical features

`Sex`, `Ascites`, `Hepatomegaly`, `Spiders`, `Edema` are text (`"F"`/`"M"`,
`"Y"`/`"N"`, etc.) — models need numbers. `pd.get_dummies` converts each
category into its own 0/1 column.

In [ ]:
categorical_cols = ["Sex", "Ascites", "Hepatomegaly", "Spiders", "Edema"]

X = pd.get_dummies(model_df[feature_cols], columns=categorical_cols, drop_first=True)
y = model_df["Stage"]
X.head()

## Train/test split

We hold out 20% of patients to evaluate the model on data it never saw
during training. `stratify=y` keeps the same Stage proportions in both
the train and test sets — important given the class imbalance, otherwise
the small `Stage 1` group could end up almost entirely in one split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## Baseline model — Logistic Regression

We start with a simple, interpretable model before trying anything fancier.
Logistic Regression is sensitive to feature scale (`Age` is in the thousands,
`Albumin` is around 3-4), so we scale everything to comparable ranges first
with `StandardScaler`, fitted only on the training set to avoid leaking test
information into the scaling.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
print("Train accuracy:", log_reg.score(X_train_scaled, y_train))
print("Test accuracy:", log_reg.score(X_test_scaled, y_test))

## Beyond accuracy — per-class performance

With only 12 `Stage 1` patients total (≈2 in the test set), overall accuracy
can look fine while the model completely fails on that class. A classification
report and confusion matrix show performance **per class**, which matters
more here than one aggregate number.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = log_reg.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
plt.show()

**Result:** accuracy = 0.34 — only modestly better than chance (25% for 4
classes). Per-class F1 ranges from 0.17 (`Stage 2`) to 0.41 (`Stage 3`).
The confusion matrix shows most errors are between **adjacent** stages
(2↔3: 16 misclassifications, 3↔4: 14) rather than distant ones (1 vs. 4 are
almost never confused) — this matches the biology: disease severity
progresses gradually, so neighboring stages have overlapping biomarker
values. That's a property of the data, not just a modeling failure — but a
linear model is also too simple to capture non-linear interactions between
biomarkers. Next: try Random Forest, which can.

## A stronger model — Random Forest

Random Forest builds many decision trees and averages their votes. It can
capture non-linear relationships and interactions between biomarkers that
Logistic Regression can't, and doesn't need feature scaling. We also set
`class_weight="balanced"` so the tiny `Stage 1` group isn't just ignored.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
print("Train accuracy:", rf.score(X_train, y_train))
print("Test accuracy:", rf.score(X_test, y_test))

In [ ]:
y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=ax)
plt.show()

**Result:** train accuracy = 1.0, test accuracy = 0.43 — a large gap, which is
**overfitting**: the model memorized the training patients instead of learning
generalizable patterns. `Stage 1` gets precision = recall = 0 despite
`class_weight="balanced"` — with only 12 total patients in that class,
there just isn't enough data for any model to learn it reliably; this is a
data limitation, not something a hyperparameter can fix.

The other classes did improve over Logistic Regression (Stage 3 F1: 0.41→0.50,
Stage 4: 0.34→0.47), but the overfitting means the 43% test accuracy is an
unstable estimate — a different train/test split could easily give a very
different number.

## Reducing overfitting + a more reliable estimate

Two changes: limit tree complexity (`max_depth`, `min_samples_leaf`) so the
model has to generalize instead of memorize, and use **5-fold cross-validation**
instead of a single 56-patient test split, which is small enough that one
split's score is noisy.

In [ ]:
from sklearn.model_selection import cross_val_score

rf_regularized = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
)

cv_scores = cross_val_score(rf_regularized, X, y, cv=5)
print("Cross-validation accuracy per fold:", cv_scores.round(3))
print("Mean:", cv_scores.mean().round(3), " Std:", cv_scores.std().round(3))

In [ ]:
rf_regularized.fit(X_train, y_train)
print("Train accuracy:", rf_regularized.score(X_train, y_train))
print("Test accuracy:", rf_regularized.score(X_test, y_test))

y_pred_reg = rf_regularized.predict(X_test)
print(classification_report(y_test, y_pred_reg))

**Result:** cross-validation gives **48.5% ± 6.8%** accuracy across 5 folds
(range 40-60%). This is the most trustworthy estimate we have — it isn't
tied to one lucky or unlucky 56-patient split. Regularizing the model
(`max_depth=5`, `min_samples_leaf=5`) also closed most of the overfitting
gap (train/test: 100%/43% → 74%/46%), and crucially let `Stage 1` actually
get predicted (recall = 1.00, though precision is still low at 0.29 — the
model now over-predicts Stage 1 sometimes, but at least finds it).

## Which biomarkers actually matter? — SHAP

Accuracy tells us *how good* the model is; SHAP tells us *why* it makes the
predictions it does. For each patient and each feature, SHAP estimates how
much that feature pushed the prediction up or down. Averaging the absolute
SHAP value per feature (across all 4 classes) gives an overall importance
ranking — more useful for a portfolio than the model's internal
"feature_importances_", because it's model-agnostic and additive.

In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(rf_regularized)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))

importance = pd.Series(mean_abs_shap, index=X_test.columns).sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
importance.plot(kind="barh", ax=ax)
ax.set_xlabel("Mean |SHAP value| (average impact on prediction)")
ax.set_title("Feature importance — Random Forest, all Stages")
plt.tight_layout()
plt.show()

**Result:** `Hepatomegaly` (enlarged liver — a physical exam finding, not a
lab test) dominates by a wide margin, followed by `Cholesterol`, `SGOT`, and
`Copper`. `Sex` and `Edema` contribute almost nothing.

Interesting contrast with Phase 1: the Cox model's top predictors of *death
risk* were Bilirubin, Prothrombin, Albumin, and Stage — different from what
best predicts the *Stage label itself* here. Predicting how sick someone is
and predicting how long they'll survive aren't the same question, even in
the same disease.

## Summary — Phase 2

- **Data:** 276/418 patients had complete data on the 15 chosen biomarkers;
  target has a real class imbalance (`Stage 1`: 12 patients vs. 94-111 for
  the rest).
- **Model comparison:** Logistic Regression (34% accuracy) → Random Forest,
  unregularized (43% test, but 100% train — badly overfit) → Random Forest,
  regularized (46% test / 74% train, 48.5% ± 6.8% via 5-fold CV). The
  regularized model is the honest number to report.
- **Biggest limitation:** `Stage 1` has too few patients (12 total) for any
  model to learn reliably — more a data problem than a modeling one.
- **Most important predictor:** `Hepatomegaly`, a simple clinical exam
  finding — outperforms every lab value.

Next: Phase 5 — polish the repo (results section in README, clean commit
history) before treating this as portfolio-ready.